## IndoBERT Embedding untuk Siamese BiLSTM

### Setup IndoBERT + CUDA untuk Kaggle

**PERSYARATAN KAGGLE:**
1. **GPU diperlukan** (tanpa fallback CPU)
2. Aktifkan di **Settings → Accelerator → GPU T4/P100**
3. Notebook akan error jika GPU tidak tersedia

**Embedding Engine:**
- Model: `indobenchmark/indobert-base-p1` 
- Output: 768 dimensi penuh
- Format: Identik dengan FastText (per-sample answers, per-IDPSJ questions/answerkeys)


In [ ]:
# Install dependencies (jalankan sekali jika error import)
# %pip install -q transformers scikit-learn torch tqdm pandas numpy

In [ ]:
import os
import ast
import random
import warnings
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

warnings.filterwarnings('ignore')

# Setup random seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Setup Kaggle Environment
DATASET_SLUG = 'siamese-data'
DATA_DIR = f'/kaggle/input/{DATASET_SLUG}'
OUT_DIR = '/kaggle/working'
os.makedirs(OUT_DIR, exist_ok=True)

# Paksa CUDA (tanpa fallback CPU)
if not torch.cuda.is_available():
    raise RuntimeError('CUDA tidak tersedia! Aktifkan GPU di Kaggle (Settings → Accelerator → GPU).')

DEVICE = 'cuda'
torch.cuda.manual_seed_all(SEED)

print("=" * 60)
print("SETUP INDOBERT EMBEDDING")
print("=" * 60)
print(f'Torch    : {torch.__version__}')
print(f'Device   : {DEVICE}')
print(f'GPU Name : {torch.cuda.get_device_name(0)}')
print(f'DATA_DIR : {DATA_DIR}')
print(f'OUT_DIR  : {OUT_DIR}')

In [ ]:
# Load dataset
INPUT_CSV = 'preprocessed.csv'
CSV_PATH = os.path.join(DATA_DIR, INPUT_CSV)

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f'File tidak ditemukan: {CSV_PATH}')

df = pd.read_csv(CSV_PATH).reset_index(drop=True)
print(f"\nLoaded CSV: {CSV_PATH}")
print(f"Total baris: {len(df)}")
print(f"Kolom: {list(df.columns)}")
df.head()

In [ ]:
# Validasi kolom yang diperlukan
TEXT_COLUMNS = {
    'questions': 'questions_clean',
    'answers': 'answer_clean',
    'answerkeys': 'answerKeys_clean',
}

for _, col in TEXT_COLUMNS.items():
    if col not in df.columns:
        raise ValueError(f"Kolom '{col}' tidak ada di {CSV_PATH}")

for c in ['IDJwb', 'IDPSJ', 'grade']:
    if c not in df.columns:
        raise ValueError(f"Kolom wajib '{c}' tidak ada di {CSV_PATH}")

print("\n=== Informasi Dataset ===")
print(f"Total baris: {len(df)}")
print(f"Kolom: {list(df.columns)}")

In [ ]:
# Parse string representation menjadi actual list (karena CSV menyimpan list sebagai string)
df['questions_list'] = df['questions_clean'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df['answerKeys_list'] = df['answerKeys_clean'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df['answer_list'] = df['answer_clean'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

### Analisis Panjang Token

**Sebelum membuat embedding, kita perlu tahu distribusi panjang token untuk menentukan `SEQ_LEN` yang optimal:**

In [ ]:
import matplotlib.pyplot as plt

def count_tokens(text_list):
    """Menghitung jumlah token dari list of strings"""
    if not text_list or len(text_list) == 0:
        return 0
    all_text = ' '.join(text_list)
    return len(all_text.split())

# Hitung panjang token untuk setiap kolom
print("=== Analisis Panjang Token ===\n")

results = {}
for col_name, col_data in [('questions', 'questions_list'), 
                            ('answerKeys', 'answerKeys_list'), 
                            ('answers', 'answer_list')]:
    token_lengths = df[col_data].apply(count_tokens)
    
    print(f"--- {col_name.upper()} ---")
    print(f"Rata-rata: {token_lengths.mean():.1f} token")
    print(f"Median: {token_lengths.median():.1f} token")
    print(f"Min: {token_lengths.min()} token")
    print(f"Max: {token_lengths.max()} token")
    print(f"Std Dev: {token_lengths.std():.1f}")
    print(f"Percentile 75%: {token_lengths.quantile(0.75):.1f} token")
    print(f"Percentile 90%: {token_lengths.quantile(0.90):.1f} token")
    print(f"Percentile 95%: {token_lengths.quantile(0.95):.1f} token")
    print(f"Percentile 99%: {token_lengths.quantile(0.99):.1f} token")
    print()
    
    results[col_name] = {
        'p95': token_lengths.quantile(0.95),
        'p99': token_lengths.quantile(0.99),
        'max': token_lengths.max()
    }

# Visualisasi distribusi
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (col_name, col_data) in enumerate([('Questions', 'questions_list'), 
                                                ('AnswerKeys', 'answerKeys_list'), 
                                                ('Answers', 'answer_list')]):
    token_lengths = df[col_data].apply(count_tokens)
    axes[idx].hist(token_lengths, bins=50, edgecolor='black')
    axes[idx].set_title(f'{col_name} Token Distribution')
    axes[idx].set_xlabel('Number of Tokens')
    axes[idx].set_ylabel('Frequency')
    axes[idx].axvline(token_lengths.mean(), color='red', linestyle='--', label=f'Mean: {token_lengths.mean():.1f}')
    axes[idx].axvline(token_lengths.quantile(0.95), color='green', linestyle='--', label=f'95%: {token_lengths.quantile(0.95):.1f}')
    axes[idx].legend()

plt.tight_layout()

print("\n" + "="*60)
print("📊 REKOMENDASI SEQ_LEN")
print("="*60)
print("\n💡 Gunakan P95-P99, BUKAN nilai maksimum!")
print("   Alasan: 1-2 outlier panjang akan menyebabkan padding berlebihan\n")

for col_name, var_name in [('questions', 'SEQ_LEN_QUESTIONS'), 
                            ('answerKeys', 'SEQ_LEN_ANSWERKEYS'), 
                            ('answers', 'SEQ_LEN_ANSWERS')]:
    p95 = int(results[col_name]['p95'])
    p99 = int(results[col_name]['p99'])
    max_val = int(results[col_name]['max'])
    
    rec_p95 = ((p95 // 5) + 1) * 5
    rec_p99 = ((p99 // 5) + 1) * 5
    
    print(f"{col_name.upper()}:")
    print(f"  ✓ Rekomendasi (95% data): {var_name} = {rec_p95}")
    print(f"  • Alternatif (99% data):  {var_name} = {rec_p99}")
    print(f"  ✗ Tidak disarankan (max): {var_name} = {max_val}")
    print()

### Load IndoBERT + Konfigurasi Embedding

**Pipeline:**
1. Load IndoBERT tokenizer & model (768-dim hidden states)
2. Encode setiap kata → subword tokens → ambil hidden state → mean-pool
3. Gunakan langsung vektor 768 dimensi tanpa PCA
4. Generate sequence embeddings (seq_len, 768)

In [ ]:
# ===== KONFIGURASI SEQ_LEN =====
# Sesuaikan dengan hasil analisis di atas

SEQ_LEN_QUESTIONS  = int(((results['questions']['p95']   // 5) + 1) * 5)
SEQ_LEN_ANSWERKEYS = int(((results['answerKeys']['p95']  // 5) + 1) * 5)
SEQ_LEN_ANSWERS    = int(((results['answers']['p95']     // 5) + 1) * 5)

BERT_MODEL_NAME = 'indobenchmark/indobert-base-p1'
TARGET_DIM = 768

print(f"SEQ_LEN_QUESTIONS  : {SEQ_LEN_QUESTIONS}")
print(f"SEQ_LEN_ANSWERKEYS : {SEQ_LEN_ANSWERKEYS}")
print(f"SEQ_LEN_ANSWERS    : {SEQ_LEN_ANSWERS}")
print(f"TARGET_DIM         : {TARGET_DIM} (full IndoBERT hidden size)")

# Load IndoBERT
print(f"\n⏳ Loading {BERT_MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME, use_fast=True)
model = AutoModel.from_pretrained(BERT_MODEL_NAME).to(DEVICE)
model.eval()
print("✅ Model loaded!\n")

def text_to_words(text_or_list):
    """Parse text/list menjadi word tokens yang konsisten"""
    if isinstance(text_or_list, list):
        # Jika sudah list dari parsing
        pieces = [str(x) for x in text_or_list if str(x).strip()]
        return ' '.join(pieces).split()
    
    if pd.isna(text_or_list):
        return []
    
    # Jika string biasa atau string yang berisi list notation
    text_str = str(text_or_list).strip()
    if text_str.startswith('[') and text_str.endswith(']'):
        try:
            parsed = ast.literal_eval(text_str)
            if isinstance(parsed, list):
                pieces = [str(x) for x in parsed if str(x).strip()]
                return ' '.join(pieces).split()
        except Exception:
            pass
    
    return text_str.split()

def word_embeddings_bert(words, max_words):
    """Encode words dengan IndoBERT, output (max_words, 768)"""
    hidden_size = 768
    vec = np.zeros((max_words, hidden_size), dtype=np.float32)
    
    if not words:
        return vec
    
    words_truncated = words[:max_words]
    
    # Tokenize words
    enc = tokenizer(
        words_truncated,
        is_split_into_words=True,
        return_tensors='pt',
        truncation=True,
        max_length=512,
        add_special_tokens=True,
    )
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    
    # Get hidden states
    with torch.no_grad():
        hidden = model(**enc).last_hidden_state[0].detach().cpu().numpy()
    
    # Map subword tokens back ke words (mean-pool)
    word_ids = tokenizer(
        words_truncated,
        is_split_into_words=True,
        truncation=True,
        max_length=512,
        add_special_tokens=True,
    ).word_ids()
    
    buckets = {}
    for tok_i, w_i in enumerate(word_ids):
        if w_i is None:
            continue
        buckets.setdefault(w_i, []).append(hidden[tok_i])
    
    for i in range(min(len(words_truncated), max_words)):
        if i in buckets:
            vec[i] = np.mean(np.stack(buckets[i], axis=0), axis=0)
    
    return vec

print("✅ IndoBERT full 768-dim embeddings aktif tanpa PCA.\n")

In [ ]:
def get_sequence_embedding_bert(texts, seq_len, desc):
    """
    Generate sequence embedding menggunakan IndoBERT full 768 dimensi.
    Output: (len(texts), seq_len, TARGET_DIM)
    """
    out = np.zeros((len(texts), seq_len, TARGET_DIM), dtype=np.float64)
    
    for i, text in enumerate(tqdm(texts, desc=desc)):
        words = text_to_words(text)
        raw = word_embeddings_bert(words, seq_len)
        mask = np.linalg.norm(raw, axis=1) > 0
        if np.any(mask):
            out[i, mask] = raw[mask].astype(np.float64)
    
    return out

print("=" * 60)
print("🚀 MEMPROSES EMBEDDING (IndoBERT full 768)")
print("=" * 60)
print(f"\nTotal data: {len(df)} baris")
print(f"Total unique IDPSJ: {df['IDPSJ'].nunique()}\n")

# ===== 1. PROSES QUESTIONS & ANSWERKEYS =====
print("=" * 60)
print("1️⃣ Memproses Questions & AnswerKeys (unique IDPSJ only)")
print("=" * 60)

unique_idpsj = df.drop_duplicates(subset='IDPSJ')[['IDPSJ', 'questions_list', 'answerKeys_list']].copy()
print(f"Memproses {len(unique_idpsj)} unique IDPSJ...")

print("\n⏳ Processing Questions...")
unique_idpsj['questions_emb'] = unique_idpsj['questions_list'].apply(
    lambda x: get_sequence_embedding_bert([x], SEQ_LEN_QUESTIONS, '')
).apply(lambda x: x[0])

print("\n⏳ Processing AnswerKeys...")
unique_idpsj['answerkeys_emb'] = unique_idpsj['answerKeys_list'].apply(
    lambda x: get_sequence_embedding_bert([x], SEQ_LEN_ANSWERKEYS, '')
).apply(lambda x: x[0])

# Merge kembali ke df
df = df.merge(unique_idpsj[['IDPSJ', 'questions_emb', 'answerkeys_emb']], on='IDPSJ', how='left')
print("✅ Questions & AnswerKeys selesai!\n")

# ===== 2. PROSES STUDENT ANSWERS =====
print("=" * 60)
print("2️⃣ Memproses Student Answers (semua baris)")
print("=" * 60)
print(f"Memproses {len(df)} answers...\n")

print("⏳ Processing Answers...")
df['answer_emb'] = df['answer_list'].apply(
    lambda x: get_sequence_embedding_bert([x], SEQ_LEN_ANSWERS, '')
).apply(lambda x: x[0])

print("✅ Student Answers selesai!\n")

# ===== 3. VERIFIKASI HASIL =====
print("=" * 60)
print("✅ SEMUA EMBEDDING SELESAI DIPROSES!")
print("=" * 60)

print(f"\nJumlah baris: {len(df)}")
print(f"\nKolom embedding yang tersedia:")
print(f"  ✓ questions_emb:  {df['questions_emb'].iloc[0].shape}")
print(f"  ✓ answerkeys_emb: {df['answerkeys_emb'].iloc[0].shape}")
print(f"  ✓ answer_emb:     {df['answer_emb'].iloc[0].shape}")

In [ ]:
# Verify sample embedding
emb_sample = df['answerkeys_emb'].iloc[0]
print("Sample embedding shape:", emb_sample.shape)
print("Min/Max values:", emb_sample.min(), emb_sample.max())
emb_sample

In [ ]:
# Proses kolom grade: konversi dari format "10,00" → integer 1-10
df['grade'] = df['grade'].astype(str).str.replace(',', '.').astype(float).round().astype(int)
df['grade'] = df['grade'].clip(lower=1, upper=10)

print("Distribusi grade setelah konversi:")
print(df['grade'].value_counts().sort_index())

In [ ]:
df = df[['IDJwb', 'IDPSJ', 'questions_emb', 'answerkeys_emb', 'answer_emb', 'grade']]

### Simpan Hasil Embedding

**Output Files:**
- `answers_emb.npy` : simpan penuh (setiap sampel unik)
- `questions_emb.npy` : simpan hanya 1 per IDPSJ (hemat storage)
- `answerkeys_emb.npy` : simpan hanya 1 per IDPSJ (hemat storage)  
- `metadata.pkl` : rekonstruksi di training menggunakan kolom `psj_idx`

In [ ]:
import pickle

print("Menyimpan hasil akhir...\n")

OUTPUT_DIR = os.path.join(OUT_DIR, "Embedding", "IndoBERT")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Stack embeddings
answers_emb = np.stack(df['answer_emb'].values)
questions_emb = np.stack(df['questions_emb'].values)
answerkeys_emb = np.stack(df['answerkeys_emb'].values)

# Create metadata dengan psj_idx mapping
metadata = df[['IDJwb', 'IDPSJ', 'grade']].copy()
idpsj_sorted = sorted(metadata['IDPSJ'].unique())
idpsj_to_idx = {psj: i for i, psj in enumerate(idpsj_sorted)}
final_metadata = metadata.copy()
final_metadata['psj_idx'] = final_metadata['IDPSJ'].map(idpsj_to_idx)

# Ambil satu baris representatif per IDPSJ
uniq_q_rows = [final_metadata.index[final_metadata['IDPSJ'] == psj][0] for psj in idpsj_sorted]
uniq_questions_emb = questions_emb[uniq_q_rows]
uniq_answerkeys_emb = answerkeys_emb[uniq_q_rows]

# Simpan paths
answers_path = os.path.join(OUTPUT_DIR, 'answers_emb.npy')
questions_path = os.path.join(OUTPUT_DIR, 'questions_emb.npy')
answerkeys_path = os.path.join(OUTPUT_DIR, 'answerkeys_emb.npy')
metadata_path = os.path.join(OUTPUT_DIR, 'metadata.pkl')

# Simpan files
np.save(answers_path, answers_emb)
np.save(questions_path, uniq_questions_emb)
np.save(answerkeys_path, uniq_answerkeys_emb)

# Save metadata dengan pickle protocol 4 (standard, hindari numpy module corruption)
with open(metadata_path, 'wb') as f:
    pickle.dump(final_metadata, f, protocol=4)

print("File tersimpan:")
print(f"  {answers_path}    {answers_emb.shape}   (per sampel)")
print(f"  {questions_path}  {uniq_questions_emb.shape}  (per IDPSJ unik)")
print(f"  {answerkeys_path} {uniq_answerkeys_emb.shape} (per IDPSJ unik)")
print(f"  {metadata_path}       {len(final_metadata)} rows  (+ kolom psj_idx)")

before_mb = (questions_emb.nbytes + answerkeys_emb.nbytes) / 1024**2
after_mb = (uniq_questions_emb.nbytes + uniq_answerkeys_emb.nbytes) / 1024**2
print(f"\nHemat storage questions+answerkeys: {before_mb:.1f} MB -> {after_mb:.1f} MB")

print("\n✅ SELESAI! IndoBERT embeddings siap untuk training.")